In [ ]:
# CELL 0 — Install, upload, merge data, train model
# Run this cell first. All subsequent cells depend on variables defined here.
!pip install shap statsmodels -q

import os, warnings, builtins
import numpy as np
import pandas as pd
from scipy import stats
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import LeaveOneOut, learning_curve
from sklearn.metrics import mean_squared_error, mean_absolute_error
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec
import shap
warnings.filterwarnings('ignore')
os.makedirs('/content/figures', exist_ok=True)

from google.colab import files
print('Please upload BOTH files:\n  1. Questionnaire.xlsx\n  2. 13_institutions_168h.xlsx')
uploaded = files.upload()

surv = pd.read_excel('Questionnaire.xlsx')
surv['TSV']  = pd.to_numeric(surv['TSV'], errors='coerce')
surv['date'] = surv['date'].astype(str).str[:10]
surv         = surv.dropna(subset=['TSV']).reset_index(drop=True)
print(f'Questionnaire loaded: {len(surv)} valid responses')

env = pd.read_excel('13_institutions_168h.xlsx')
env['datetime'] = pd.to_datetime(env['datetime'])
env['date']     = env['datetime'].dt.date.astype(str)
env['hour']     = env['datetime'].dt.hour
print(f'Environmental data: {env.shape[0]} rows, {env["institution_id"].nunique()} institutions')

surv_mean = (surv
             .groupby(['institution_id', 'date', 'hour'])['TSV']
             .agg(TSV_survey='mean', n_resp='count')
             .reset_index())

env_sub = (env[[
    'institution_id', 'date', 'hour', 'district',
    'window_wall_ratio', 'TA_mean', 'RH_mean', 'TG_mean',
    'Total_Radiation_mean', 'Wind_Speed_mean',
    'pm25_mean', 'co2_mean', 'Noise_mean']]
    .rename(columns={
        'window_wall_ratio':    'WWR',
        'Total_Radiation_mean': 'Rad_mean',
        'Wind_Speed_mean':      'WS_mean',
        'pm25_mean':            'PM25_mean',
        'co2_mean':             'CO2_mean'}))

matched = (surv_mean
           .merge(env_sub, on=['institution_id', 'date', 'hour'], how='inner')
           .dropna()
           .reset_index(drop=True))
print(f'Merged dataset: n={len(matched)} institutional time-point means')

# --- Dual-layer descriptive statistics ---
tsv_individual  = surv['TSV'].values
tsv_inst_means  = matched['TSV_survey'].values
pct_neutral_A   = ((tsv_individual >= -0.5) & (tsv_individual <= 0.5)).mean() * 100
pct_neutral_B   = ((tsv_inst_means  >= -0.5) & (tsv_inst_means  <= 0.5)).mean() * 100

print(f'\n[Layer A] Individual responses (n={len(tsv_individual)})')
print(f'  Neutral zone (TSV=0): {pct_neutral_A:.1f}%')
print(f'[Layer B] Inst. time-point means (n={len(tsv_inst_means)})')
print(f'  Neutral zone [-0.5,+0.5]: {pct_neutral_B:.1f}%')

# --- Feature configuration ---
FEATS = ['TA_mean','RH_mean','TG_mean','Rad_mean',
         'WS_mean','PM25_mean','CO2_mean','WWR','Noise_mean']
FEAT_LABELS = {
    'TA_mean':    'Air Temp $T_a$ (\u00b0C)',
    'RH_mean':    'Relative Humidity (%)',
    'TG_mean':    'Globe Temp $T_G$ (\u00b0C)',
    'Rad_mean':   'Solar Radiation (W/m\u00b2)',
    'WS_mean':    'Wind Speed (m/s)',
    'PM25_mean':  '$PM_{2.5}$ (\u03bcg/m\u00b3)',
    'CO2_mean':   '$CO_2$ (ppm)',
    'WWR':        'Window-Wall Ratio',
    'Noise_mean': 'Noise Level (dB)',
}
PALETTE = {
    'blue':'#0072B2','red':'#D55E00','orange':'#E69F00',
    'green':'#009E73','gray':'#999999','purple':'#CC79A7',
}
institutions = sorted(matched['institution_id'].unique())
INST_COLORS  = plt.cm.tab20(np.linspace(0, 1, len(institutions)))

X      = matched[FEATS].values
y      = matched['TSV_survey'].values
scaler = StandardScaler()
X_sc   = scaler.fit_transform(X)

# --- LOO cross-validation ---
print('\nRunning LOO cross-validation...')
loo   = LeaveOneOut()
y_loo = np.zeros(len(y))
for tr_idx, te_idx in loo.split(X_sc):
    m = GradientBoostingRegressor(
        n_estimators=100, max_depth=3,
        learning_rate=0.1, random_state=42)
    m.fit(X_sc[tr_idx], y[tr_idx])
    y_loo[te_idx] = m.predict(X_sc[te_idx])

LOO_R2    = 1 - np.sum((y-y_loo)**2) / np.sum((y-y.mean())**2)
LOO_RMSE  = np.sqrt(mean_squared_error(y, y_loo))
LOO_MAE   = mean_absolute_error(y, y_loo)
residuals = y - y_loo
sw_stat, sw_p = stats.shapiro(residuals)
matched['TSV_pred_loo'] = y_loo
matched['residuals']    = residuals

print(f'  LOO R2={LOO_R2:.4f}  RMSE={LOO_RMSE:.4f}  MAE={LOO_MAE:.4f}  SW p={sw_p:.4f}')

# --- Full-sample model + SHAP ---
print('Training full-sample model and computing SHAP...')
model_full = GradientBoostingRegressor(
    n_estimators=100, max_depth=3,
    learning_rate=0.1, random_state=42)
model_full.fit(X_sc, y)
explainer          = shap.TreeExplainer(model_full)
shap_values        = explainer.shap_values(X_sc)
mean_abs_shap      = np.abs(shap_values).mean(axis=0)
mean_abs_shap_norm = mean_abs_shap / mean_abs_shap.sum()
shap_order         = np.argsort(mean_abs_shap)
shap_order_desc    = shap_order[::-1]

print('\nSHAP feature importance (normalised, for Table 3):')
print(f'  {"Feature":<12}  {"Raw":>8}  {"Norm":>8}')
for i in shap_order_desc:
    print(f'  {FEATS[i]:<12}  {mean_abs_shap[i]:>8.4f}  '
          f'{mean_abs_shap_norm[i]:>8.4f}')

# --- Font setup ---
import subprocess, glob, shutil
import matplotlib as mpl
import matplotlib.font_manager as fm
subprocess.run(['apt-get','install','-y','-q','msttcorefonts'],
               check=False, capture_output=True)
mpl_font_dir = os.path.join(
    os.path.dirname(mpl.__file__), 'mpl-data', 'fonts', 'ttf')
for src_dir in ['/usr/share/fonts','/usr/local/share/fonts',
                os.path.expanduser('~/.fonts')]:
    for fp in glob.glob(
            os.path.join(src_dir,'**','*.tt[fc]'), recursive=True):
        dest = os.path.join(mpl_font_dir, os.path.basename(fp))
        if not os.path.exists(dest):
            try: shutil.copy2(fp, dest)
            except: pass
for fp in (glob.glob(os.path.join(mpl.get_cachedir(),'*.json')) +
           glob.glob(os.path.join(mpl.get_cachedir(),'fontlist*.cache')) +
           glob.glob(os.path.join(mpl.get_cachedir(),'*.pkl'))):
    try: os.remove(fp)
    except: pass
fm._load_fontmanager(try_read_cache=False)
_tnr = any('times new roman' in f.name.lower()
           for f in fm.fontManager.ttflist)
_font = 'Times New Roman' if _tnr else 'DejaVu Serif'
print(f'Font: {_font}')

plt.rcParams.update({
    'font.family':'serif',
    'font.serif':[_font,'Times New Roman','DejaVu Serif','serif'],
    'font.size':10, 'axes.titlesize':11, 'axes.labelsize':10,
    'axes.linewidth':0.8, 'xtick.labelsize':9, 'ytick.labelsize':9,
    'legend.fontsize':8.5, 'mathtext.fontset':'stix',
    'figure.dpi':150, 'savefig.dpi':600,
})

def save_fig(fig, name):
    for fmt in ['png','tiff']:
        fig.savefig(f'/content/figures/{name}.{fmt}',
                    dpi=600, bbox_inches='tight', facecolor='white')
    plt.close(fig)
    print(f'  Saved: {name}')

print('\nCell 0 complete.')


Please upload BOTH files:
  1. Questionnaire.xlsx
  2. 13_institutions_168h.xlsx


Saving 13_institutions_168h.xlsx to 13_institutions_168h.xlsx
Questionnaire loaded: 234 valid responses
Environmental data: 2184 rows, 13 institutions
Merged dataset: n=78 institutional time-point means

[Layer A] Individual responses (n=234)
  Neutral zone (TSV=0): 38.9%
[Layer B] Inst. time-point means (n=78)
  Neutral zone [-0.5,+0.5]: 52.6%

Running LOO cross-validation...
  LOO R2=0.3734  RMSE=0.6316  MAE=0.4874  SW p=0.7816
Training full-sample model and computing SHAP...

SHAP feature importance (normalised, for Table 3):
  Feature            Raw      Norm
  Rad_mean        0.3048    0.3174
  WWR             0.1724    0.1796
  PM25_mean       0.1349    0.1404
  CO2_mean        0.1063    0.1107
  TG_mean         0.0921    0.0959
  RH_mean         0.0704    0.0734
  TA_mean         0.0413    0.0430
  Noise_mean      0.0255    0.0266
  WS_mean         0.0125    0.0130
Font: DejaVu Serif

Cell 0 complete.


In [ ]:
# CELL 1 (REVISED) — Fig. 2: Indoor Environmental Parameter Distributions
# Data source: full 168-hour monitoring records (n=2,184 hourly records)
# This correctly represents the overall indoor thermal environment
# across all 13 institutions during the study period.

import pandas as pd
import numpy as np

env_full = pd.read_excel('13_institutions_168h.xlsx')

panels = [
    ('TA_mean',              'Air Temperature $T_a$ (\u00b0C)',
     PALETTE['blue'],   None),
    ('RH_mean',              'Relative Humidity (%)',
     PALETTE['orange'], None),
    ('Total_Radiation_mean', 'Solar Radiation (W/m\u00b2)',
     PALETTE['red'],    None),
    ('pm25_mean',            '$PM_{2.5}$ Concentration (\u03bcg/m\u00b3)',
     PALETTE['green'],  None),
]

fig, axes = plt.subplots(2, 2, figsize=(11, 8))
axes = axes.flatten()

for ax, (col, xlabel, color, _) in zip(axes, panels):
    vals = env_full[col].dropna().values
    ax.hist(vals, bins=30, color=color, alpha=0.72,
            edgecolor='white', linewidth=0.6)
    ax.axvline(vals.mean(), color='black', linewidth=1.8,
               linestyle='--',
               label=f'Mean = {vals.mean():.1f}')
    ax.axvline(np.median(vals), color=PALETTE['gray'],
               linewidth=1.3, linestyle=':',
               label=f'Median = {np.median(vals):.1f}')
    ax.set_xlabel(xlabel, fontsize=10)
    ax.set_ylabel(
        'Frequency  (n = 2,184 hourly records)',
        fontsize=9)
    ax.set_title(
        f'{xlabel.split("(")[0].strip()}\n'
        f'Mean={vals.mean():.1f},  SD={vals.std():.1f},  '
        f'Range=[{vals.min():.1f}, {vals.max():.1f}]',
        fontweight='bold')
    ax.legend(framealpha=0.88, fontsize=8.5)
    ax.grid(axis='y', alpha=0.22, linewidth=0.5)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

fig.suptitle(
    'Fig. 2  Indoor Environmental Parameter Distributions\n'
    '(168-hour continuous monitoring; n = 2,184 hourly records; '
    '13 institutions; Guiyang Summer 2025)',
    fontsize=11.5, fontweight='bold', y=1.01)
plt.tight_layout()
save_fig(fig, 'fig2_env_distributions')

print('\nFig. 2 key statistics (n=2,184 hourly records):')
for col, label, _, _ in panels:
    v = env_full[col].dropna().values
    print(f'  {label.split("(")[0].strip():<22}: '
          f'mean={v.mean():.2f}, SD={v.std():.2f}, '
          f'range=[{v.min():.2f}, {v.max():.2f}]')

  Saved: fig2_env_distributions

Fig. 2 key statistics (n=2,184 hourly records):
  Air Temperature $T_a$ : mean=26.23, SD=1.67, range=[21.53, 31.30]
  Relative Humidity     : mean=73.98, SD=6.44, range=[49.60, 90.00]
  Solar Radiation       : mean=188.17, SD=189.35, range=[0.00, 1241.20]
  $PM_{2.5}$ Concentration: mean=13.14, SD=7.19, range=[0.66, 34.70]


In [ ]:
# CELL 2  — Fig. 3: Survey TSV Distribution and Institution Comparison
fig = plt.figure(figsize=(14, 8.5))
gs  = gridspec.GridSpec(1, 2, width_ratios=[1, 2.2],
                        figure=fig, wspace=0.28)
ax1 = fig.add_subplot(gs[0])
ax2 = fig.add_subplot(gs[1])

# --- Panel A ---
tsv_raw   = surv['TSV'].values
bins      = np.arange(-2.5, 3.6, 1.0)
counts, _ = np.histogram(tsv_raw, bins=bins)
ax1.bar(bins[:-1]+0.5, counts, width=0.82,
        color=PALETTE['blue'], alpha=0.70,
        edgecolor='white', linewidth=0.5)
ax1.axvspan(-0.5, 0.5, color=PALETTE['green'], alpha=0.13,
            label='Neutral zone (\u22120.5 to +0.5)')
ax1.axvline(tsv_raw.mean(), color=PALETTE['red'],
            linewidth=2.0, linestyle='--',
            label=f'Mean = {tsv_raw.mean():.2f}')
ax1.set_xlabel('Thermal Sensation Vote (TSV)', fontsize=10)
ax1.set_ylabel('Count', fontsize=10)
ax1.set_xticks([-2,-1,0,1,2,3])
ax1.set_xticklabels(
    ['\u22122\n(Very\nCool)', '\u22121\n(Cool)', '0\n(Neutral)',
     '+1\n(Sl.\nWarm)', '+2\n(Warm)', '+3\n(Hot)'], fontsize=8)
ax1.set_title(
    f'(A) Overall TSV Distribution\n'
    f'n = 234 individual responses;  '
    f'Neutral zone = {pct_neutral_A:.1f}%',
    fontweight='bold', pad=8)
ax1.legend(framealpha=0.88, fontsize=8.5)
ax1.grid(axis='y', alpha=0.2)
ax1.spines['top'].set_visible(False)
ax1.spines['right'].set_visible(False)

# --- Panel B ---
inst_means_tsv = matched.groupby('institution_id')['TSV_survey'].mean()
top4    = set(inst_means_tsv.nlargest(4).index)
bp_data = [surv[surv['institution_id']==i]['TSV'].values
           for i in institutions]
bp = ax2.boxplot(
    bp_data, patch_artist=True,
    medianprops=dict(color='black', linewidth=2.2),
    flierprops=dict(marker='o', markersize=3.5,
                    markerfacecolor=PALETTE['gray'],
                    markeredgewidth=0, alpha=0.55),
    whiskerprops=dict(linewidth=1.2),
    capprops=dict(linewidth=1.2))
for patch, inst in zip(bp['boxes'], institutions):
    patch.set_facecolor(
        PALETTE['red'] if inst in top4 else PALETTE['blue'])
    patch.set_alpha(0.65)
for j, inst in enumerate(institutions, start=1):
    ax2.plot(j, inst_means_tsv[inst],
             marker='D', color='black', markersize=5.5, zorder=5,
             label='Institution survey mean (observed)' if j==1 else '')
ax2.axhline( 0.5, color=PALETTE['orange'], linewidth=1.3,
             linestyle='--', alpha=0.80,
             label='Comfort threshold \u00b10.5')
ax2.axhline(-0.5, color=PALETTE['orange'], linewidth=1.3,
             linestyle='--', alpha=0.80)
ax2.axhline(0, color=PALETTE['gray'], linewidth=0.8,
            linestyle=':', alpha=0.45)
ax2.set_xticks(range(1,14))
ax2.set_xticklabels(
    [f'Inst\n{i}' for i in institutions], fontsize=9)
ax2.set_ylabel('Thermal Sensation Vote (TSV)', fontsize=10)
top4_str = ', '.join(f'Inst {i}' for i in sorted(top4))
ax2.set_title(
    f'(B) TSV by Institution  (n = 18 responses each)\n'
    f'Red = highest-TSV facilities: {top4_str}',
    fontweight='bold', pad=8)
ax2.legend(handles=[
    mpatches.Patch(facecolor=PALETTE['red'],  alpha=0.65,
                   label=f'High-TSV institutions ({top4_str})'),
    mpatches.Patch(facecolor=PALETTE['blue'], alpha=0.65,
                   label='Other institutions'),
    plt.Line2D([0],[0], marker='D', color='black',
               linestyle='None', markersize=5.5,
               label='Institution survey mean (observed)'),
    plt.Line2D([0],[0], color=PALETTE['orange'],
               linewidth=1.3, linestyle='--',
               label='Comfort threshold \u00b10.5'),
], fontsize=8.5, loc='upper right', framealpha=0.88)
ax2.grid(axis='y', alpha=0.20)
ax2.spines['top'].set_visible(False)
ax2.spines['right'].set_visible(False)

# Precise spacing: top=0.85 pushes subplots down,
# creating clear gap between suptitle and panel titles.
# bottom=0.15 gives enough room for x-axis labels + note.
plt.subplots_adjust(top=0.85, bottom=0.15,
                    left=0.07, right=0.97, wspace=0.28)

# Suptitle sits in top 15% reserved space
fig.suptitle(
    'Fig. 3  Survey TSV: Overall Distribution '
    'and Institution-Level Comparison\n'
    '(Summer 2025; 13 elderly care facilities; Guiyang, China)',
    fontsize=11.5, fontweight='bold', y=0.98)

# Note sits in bottom 15% reserved space, well above x-axis labels
fig.text(
    0.5, 0.02,
    f'Note: Neutral zone proportions differ by analysis unit \u2014 '
    f'{pct_neutral_A:.1f}% of individual responses (TSV = 0) vs. '
    f'{pct_neutral_B:.1f}% of institutional time-point means '
    f'(\u22120.5 \u2264 TSV \u2264 +0.5), '
    f'due to averaging of integer votes to continuous means.',
    ha='center', fontsize=9.5, style='italic', color='#444444')

save_fig(fig, 'fig3_tsv_distribution')

  Saved: fig3_tsv_distribution


In [ ]:
# CELL 3 — Fig. 4: LOO-CV Predicted vs. Observed TSV
fig, ax = plt.subplots(figsize=(6.8, 6.5))
for j, inst in enumerate(institutions):
    mask = matched['institution_id'] == inst
    ax.scatter(matched.loc[mask,'TSV_survey'],
               matched.loc[mask,'TSV_pred_loo'],
               color=INST_COLORS[j], s=68, alpha=0.82,
               edgecolors='white', linewidths=0.4,
               label=f'Inst {inst}', zorder=3)
y_np = np.asarray(y); y_loo_np = np.asarray(y_loo)
lo = builtins.min(y_np.min(), y_loo_np.min()) - 0.30
hi = builtins.max(y_np.max(), y_loo_np.max()) + 0.30
ax.plot([lo,hi],[lo,hi], color='black', linewidth=1.6,
        linestyle='--', label='1:1 perfect agreement', zorder=2)
x_line = np.linspace(lo, hi, 200)
ax.fill_between(x_line,
                x_line - 1.96*LOO_RMSE,
                x_line + 1.96*LOO_RMSE,
                alpha=0.09, color=PALETTE['gray'],
                label=f'\u00b11.96\u00d7RMSE (\u00b1{1.96*LOO_RMSE:.2f} TSV)')
ax.set_xlim(lo,hi); ax.set_ylim(lo,hi)
ax.set_xlabel('Observed Institutional Mean TSV (Survey)', fontsize=10)
ax.set_ylabel('Predicted Institutional Mean TSV (LOO-CV)', fontsize=10)
ax.set_title(
    'Fig. 4  LOO Cross-Validation: Predicted vs. Observed Institutional TSV\n'
    'Gradient Boosting Regression;  n = 78 time-point means',
    fontweight='bold')
ax.text(0.04, 0.97,
        f'LOO R\u00b2   = {LOO_R2:.4f}\n'
        f'LOO RMSE = {LOO_RMSE:.4f}\n'
        f'LOO MAE  = {LOO_MAE:.4f}\n'
        f'Residual mean = {residuals.mean():.3f}\n'
        f'Shapiro-Wilk p = {sw_p:.3f}',
        transform=ax.transAxes, fontsize=8.8,
        va='top', ha='left', family='monospace',
        bbox=dict(boxstyle='round,pad=0.4',
                  facecolor='lightyellow',
                  edgecolor=PALETTE['gray'], alpha=0.92))
ax.legend(fontsize=7.5, loc='lower right', ncol=2,
          framealpha=0.88, handlelength=1.2)
ax.grid(alpha=0.20, linewidth=0.5)
ax.set_aspect('equal', adjustable='box')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
save_fig(fig, 'fig4_loo_predicted_vs_observed')


  Saved: fig4_loo_predicted_vs_observed


In [ ]:
# CELL 4 — Fig. 5: Residual Diagnostics (3-panel)
fig, axes = plt.subplots(1, 3, figsize=(14, 4.8))

ax = axes[0]
ax.scatter(y_loo, residuals, color=PALETTE['blue'],
           alpha=0.68, s=50, edgecolors='white', linewidths=0.4)
ax.axhline(0, color=PALETTE['red'], linewidth=1.8,
           linestyle='--', label='Zero line')
ax.axhline( 1.96*LOO_RMSE, color=PALETTE['orange'],
            linewidth=1.1, linestyle=':',
            label=f'+1.96 SD = +{1.96*LOO_RMSE:.2f}')
ax.axhline(-1.96*LOO_RMSE, color=PALETTE['orange'],
            linewidth=1.1, linestyle=':',
            label=f'\u22121.96 SD = \u2212{1.96*LOO_RMSE:.2f}')
ax.set_xlabel('Fitted Values (LOO Predicted TSV)', fontsize=10)
ax.set_ylabel('Residuals  (Observed \u2212 Predicted)', fontsize=10)
ax.set_title(
    f'(A) Residuals vs. Fitted\n'
    f'Mean = {residuals.mean():.3f},  SD = {residuals.std():.3f}',
    fontweight='bold')
ax.legend(fontsize=8.2)
ax.grid(alpha=0.20)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

ax = axes[1]
ax.hist(residuals, bins=16, density=True,
        color=PALETTE['blue'], alpha=0.68,
        edgecolor='white', linewidth=0.5)
x_fit = np.linspace(residuals.min()-0.4, residuals.max()+0.4, 300)
ax.plot(x_fit,
        stats.norm.pdf(x_fit, residuals.mean(), residuals.std()),
        color=PALETTE['red'], linewidth=2.2,
        label='Normal distribution fit')
ax.set_xlabel('Residuals', fontsize=10)
ax.set_ylabel('Density', fontsize=10)
ax.set_title(
    f'(B) Residual Distribution\n'
    f'Shapiro-Wilk:  W = {sw_stat:.3f},  p = {sw_p:.3f}',
    fontweight='bold')
ax.legend(fontsize=8.2)
ax.grid(alpha=0.20)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

ax = axes[2]
(osm,osr),(slope,intercept,r_qq) = stats.probplot(
    residuals, dist='norm')
ax.scatter(osm, osr, color=PALETTE['blue'], s=40,
           alpha=0.75, edgecolors='white', linewidths=0.3)
x_qq = np.array([osm.min(), osm.max()])
ax.plot(x_qq, slope*x_qq+intercept,
        color=PALETTE['red'], linewidth=2.0, linestyle='--',
        label=f'Reference line  (r = {r_qq:.3f})')
ax.set_xlabel('Theoretical Quantiles (Normal)', fontsize=10)
ax.set_ylabel('Sample Quantiles (Residuals)', fontsize=10)
ax.set_title(
    f'(C) Normal Q\u2013Q Plot\nPearson r = {r_qq:.3f}',
    fontweight='bold')
ax.legend(fontsize=8.2)
ax.grid(alpha=0.20)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

fig.suptitle(
    'Fig. 5  Residual Diagnostics \u2014 '
    'Gradient Boosting LOO Cross-Validation\n'
    '(n = 78 institutional time-point means)',
    fontsize=11.5, fontweight='bold', y=1.02)
plt.tight_layout()
save_fig(fig, 'fig5_residual_diagnostics')


  Saved: fig5_residual_diagnostics


In [ ]:
# CELL 5 — Fig. 6: SHAP Global Bar Chart  +  Fig. 7: Beeswarm
feat_labels_list = [FEAT_LABELS[f] for f in FEATS]

# --- Fig. 6 ---
fig, ax = plt.subplots(figsize=(8.5, 5.8))
bar_colors = [
    PALETTE['red'] if mean_abs_shap[i] >= sorted(mean_abs_shap)[-2]
    else PALETTE['blue']
    for i in shap_order]
ax.barh(range(len(FEATS)), mean_abs_shap[shap_order],
        color=bar_colors, alpha=0.80,
        edgecolor='white', height=0.68)
ax.set_yticks(range(len(FEATS)))
ax.set_yticklabels(
    [feat_labels_list[i] for i in shap_order], fontsize=10)
ax.set_xlabel(
    'Mean |SHAP Value|  \u2014  Raw Global Feature Importance',
    fontsize=10)
for i,(idx,val) in enumerate(
        zip(shap_order, mean_abs_shap[shap_order])):
    ax.text(val+0.003, i, f'{val:.3f}',
            va='center', ha='left', fontsize=8.8)
cum_top2 = (mean_abs_shap[shap_order_desc[:2]].sum() /
            mean_abs_shap.sum() * 100)
ax.set_title(
    'Fig. 6  SHAP Global Feature Importance\n'
    '(Full-sample model; n = 78; top-2 predictors highlighted in red;\n'
    ' x-axis = raw mean|SHAP|; normalised values in Table 3)',
    fontweight='bold')
ax.text(0.38, 0.02,
        f'Top-2 features account for\n{cum_top2:.1f}% of total importance',
        transform=ax.transAxes, fontsize=8.5,
        ha='left', va='bottom',
        bbox=dict(boxstyle='round', facecolor='lightyellow',
                  edgecolor=PALETTE['gray'], alpha=0.92))
ax.legend(handles=[
    mpatches.Patch(fc=PALETTE['red'],  alpha=0.80,
                   label='Top-2 dominant predictors'),
    mpatches.Patch(fc=PALETTE['blue'], alpha=0.80,
                   label='Other features'),
], loc='lower right', fontsize=8.8)
ax.set_xlim(0, mean_abs_shap.max()*1.22)
ax.grid(axis='x', alpha=0.22, linewidth=0.5)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
save_fig(fig, 'fig6_shap_bar')

print('\nTable 3 (normalised SHAP — use in paper):')
for rank,i in enumerate(shap_order_desc,1):
    print(f'  {rank}. {FEATS[i]:<12} '
          f'raw={mean_abs_shap[i]:.4f}  '
          f'norm={mean_abs_shap_norm[i]:.4f}')

# --- Fig. 7 ---
fig, ax = plt.subplots(figsize=(10.5, 6.2))
for i, feat_idx in enumerate(shap_order):
    fv = X_sc[:, feat_idx]
    sv = shap_values[:, feat_idx]
    nv = (fv-fv.min())/(fv.max()-fv.min()+1e-9)
    rng = np.random.RandomState(feat_idx+7)
    jitter = rng.uniform(-0.22, 0.22, size=len(sv))
    sc = ax.scatter(sv, i+jitter, c=nv, cmap='RdBu_r',
                    vmin=0, vmax=1, alpha=0.68, s=30, zorder=3)
ax.axvline(0, color='black', linewidth=1.0, zorder=2)
ax.set_yticks(range(len(FEATS)))
ax.set_yticklabels(
    [feat_labels_list[i] for i in shap_order], fontsize=10)
ax.set_xlabel(
    'SHAP Value  (Impact on Predicted Institutional Mean TSV)',
    fontsize=10)
ax.set_title(
    'Fig. 7  SHAP Dot Plot (Beeswarm)\n'
    '(Each point = 1 institutional time-point;  '
    'colour = normalised feature value)',
    fontweight='bold')
cbar = plt.colorbar(sc, ax=ax, fraction=0.024, pad=0.02)
cbar.set_label(
    'Feature Value\n(Blue = Low  |  Red = High)', fontsize=8.5)
cbar.set_ticks([0,0.5,1])
cbar.set_ticklabels(['Low','Mid','High'], fontsize=8)
ax.grid(axis='x', alpha=0.20, linewidth=0.5)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
save_fig(fig, 'fig7_shap_beeswarm')


  Saved: fig6_shap_bar

Table 3 (normalised SHAP — use in paper):
  1. Rad_mean     raw=0.3048  norm=0.3174
  2. WWR          raw=0.1724  norm=0.1796
  3. PM25_mean    raw=0.1349  norm=0.1404
  4. CO2_mean     raw=0.1063  norm=0.1107
  5. TG_mean      raw=0.0921  norm=0.0959
  6. RH_mean      raw=0.0704  norm=0.0734
  7. TA_mean      raw=0.0413  norm=0.0430
  8. Noise_mean   raw=0.0255  norm=0.0266
  9. WS_mean      raw=0.0125  norm=0.0130
  Saved: fig7_shap_beeswarm


In [ ]:
# CELL 6 — Fig. 8: SHAP Dependence Plot — Solar Radiation and WWR
from statsmodels.nonparametric.smoothers_lowess import lowess

rad_idx  = FEATS.index('Rad_mean')
rad_orig = scaler.inverse_transform(X_sc)[:, rad_idx]
wwr_orig = matched['WWR'].values
wwr_p33  = np.percentile(wwr_orig, 33)
wwr_p67  = np.percentile(wwr_orig, 67)

inst_ids      = matched['institution_id'].values
inst_shap_rad = {inst: np.mean(shap_values[inst_ids==inst, rad_idx])
                 for inst in institutions}
inst_wwr_val  = {inst: wwr_orig[inst_ids==inst][0]
                 for inst in institutions}
shap_by_inst  = np.array([inst_shap_rad[i] for i in institutions])
wwr_by_inst   = np.array([inst_wwr_val[i]  for i in institutions])
rho, p_spear  = stats.spearmanr(wwr_by_inst, shap_by_inst)
print(f'Spearman: rho={rho:.3f}, p={p_spear:.3f} (n={len(institutions)})')

fig, axes = plt.subplots(1, 2, figsize=(14, 5.8))

# --- Panel A ---
ax = axes[0]
tertile_masks  = [
    wwr_orig <= wwr_p33,
    (wwr_orig > wwr_p33) & (wwr_orig <= wwr_p67),
    wwr_orig > wwr_p67,
]
tertile_labels = [
    f'Low WWR (\u2264{wwr_p33:.2f})',
    f'Mid WWR ({wwr_p33:.2f}\u2013{wwr_p67:.2f})',
    f'High WWR (>{wwr_p67:.2f})',
]
tertile_colors = [PALETTE['blue'],PALETTE['orange'],PALETTE['red']]
sc = ax.scatter(
    rad_orig, shap_values[:, rad_idx],
    c=wwr_orig, cmap='RdYlBu_r',
    vmin=wwr_orig.min(), vmax=wwr_orig.max(),
    alpha=0.50, s=40, edgecolors='white', linewidths=0.3, zorder=2)
for mask,lbl,clr in zip(tertile_masks,tertile_labels,tertile_colors):
    if mask.sum() >= 6:
        trend = lowess(
            shap_values[mask, rad_idx], rad_orig[mask],
            frac=0.55, it=2, return_sorted=True)
        ax.plot(trend[:,0], trend[:,1], color=clr,
                linewidth=2.2, zorder=4, label=f'LOWESS: {lbl}')
ax.axvline(300, color=PALETTE['gray'], linewidth=1.2,
           linestyle='--', alpha=0.75,
           label='~300 W/m\u00b2  reference')
ax.axhline(0, color=PALETTE['gray'], linewidth=0.8,
           linestyle=':', alpha=0.55)
cbar = plt.colorbar(sc, ax=ax, fraction=0.045, pad=0.02)
cbar.set_label('Window-Wall Ratio (WWR)', fontsize=8.8)
ax.set_xlabel('Solar Radiation (W/m\u00b2)', fontsize=10)
ax.set_ylabel('SHAP Value for Solar Radiation', fontsize=10)
ax.set_title(
    '(A) SHAP(Solar Radiation) vs. Solar Radiation\n'
    'Coloured by WWR; separate LOWESS per WWR tertile',
    fontweight='bold')
ax.legend(fontsize=7.8, loc='upper left')
ax.grid(alpha=0.20)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

# --- Panel B ---
ax = axes[1]
colors_inst = [
    PALETTE['red'] if w > wwr_p67
    else (PALETTE['orange'] if w > wwr_p33 else PALETTE['blue'])
    for w in wwr_by_inst]
ax.scatter(wwr_by_inst, shap_by_inst, c=colors_inst,
           s=90, alpha=0.85, edgecolors='white',
           linewidths=0.5, zorder=4)
for i,inst in enumerate(institutions):
    ax.annotate(f'I{inst}',
                (wwr_by_inst[i], shap_by_inst[i]),
                fontsize=7, ha='center', va='bottom',
                xytext=(0,4), textcoords='offset points')
z    = np.polyfit(wwr_by_inst, shap_by_inst, 1)
xfit = np.linspace(wwr_by_inst.min()-0.03,
                   wwr_by_inst.max()+0.03, 100)
ax.plot(xfit, np.poly1d(z)(xfit),
        color=PALETTE['gray'], linewidth=1.5,
        linestyle='--', alpha=0.7, label='OLS trend')
ax.axhline(0, color=PALETTE['gray'], linewidth=0.8,
           linestyle=':', alpha=0.55)
sig = ('*' if p_spear<0.05 else
       '\u2020' if p_spear<0.10 else 'n.s.')
ax.text(0.97, 0.97,
        f'Spearman correlation\n'
        f'\u03c1 = {rho:.3f},  p = {p_spear:.3f}  {sig}\n'
        f'(n = {len(institutions)} institutions)',
        transform=ax.transAxes, fontsize=8.8,
        va='top', ha='right',
        bbox=dict(boxstyle='round', facecolor='lightyellow',
                  edgecolor=PALETTE['gray'], alpha=0.92))
ax.set_xlabel('Window-Wall Ratio (WWR)', fontsize=10)
ax.set_ylabel(
    'Institution Mean SHAP(Solar Radiation)', fontsize=10)
ax.set_title(
    f'(B) Institution-Level: WWR vs. Mean SHAP(Solar Radiation)\n'
    f'(n = {len(institutions)} institutions;  '
    f'Spearman \u03c1 = {rho:.3f},  p = {p_spear:.3f})',
    fontweight='bold')
ax.legend(fontsize=8.5)
ax.grid(alpha=0.20)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

fig.suptitle(
    'Fig. 8  SHAP Dependence Plot: '
    'Solar Radiation and Window-Wall Ratio\n'
    '(Urban microclimate \u2192 building envelope \u2192 '
    'indoor thermal sensation pathway)',
    fontsize=11.5, fontweight='bold', y=1.02)
plt.tight_layout()
save_fig(fig, 'fig8_shap_dependence')


Spearman: rho=-0.341, p=0.255 (n=13)
  Saved: fig8_shap_dependence


In [ ]:
# CELL 7 — Fig. 9 : Learning Curve + pack & download
print('Computing learning curve (LOO-CV)...')
y_array     = matched['TSV_survey'].values
train_fracs = np.linspace(0.30, 1.0, 10)
tr_sizes, tr_scores, val_scores = learning_curve(
    GradientBoostingRegressor(
        n_estimators=100, max_depth=3,
        learning_rate=0.1, random_state=42),
    X_sc, y_array,
    train_sizes=train_fracs,
    cv=LeaveOneOut(),
    scoring='neg_mean_squared_error',
    n_jobs=-1)
tr_mse  = -tr_scores
val_mse = -val_scores

fig, ax = plt.subplots(figsize=(7.8, 5.5))
ax.plot(tr_sizes, tr_mse.mean(axis=1),
        color=PALETTE['blue'], marker='o', markersize=6.5,
        linewidth=1.8, label='Training MSE', zorder=3)
ax.fill_between(
    tr_sizes,
    tr_mse.mean(axis=1)-tr_mse.std(axis=1),
    tr_mse.mean(axis=1)+tr_mse.std(axis=1),
    alpha=0.14, color=PALETTE['blue'])
ax.plot(tr_sizes, val_mse.mean(axis=1),
        color=PALETTE['orange'], marker='s', markersize=6.5,
        linewidth=1.8, label='LOO Validation MSE', zorder=3)
ax.fill_between(
    tr_sizes,
    val_mse.mean(axis=1)-val_mse.std(axis=1),
    val_mse.mean(axis=1)+val_mse.std(axis=1),
    alpha=0.14, color=PALETTE['orange'])
ax.axhline(LOO_RMSE**2, color=PALETTE['red'],
           linewidth=1.6, linestyle='--',
           label=f'Final LOO MSE = {LOO_RMSE**2:.4f}  '
                 f'(RMSE = {LOO_RMSE:.4f})')
ax.set_xlabel('Training Set Size (samples)', fontsize=10)
ax.set_ylabel('Mean Squared Error (MSE)', fontsize=10)
ax.set_title(
    'Fig. 9  Learning Curve \u2014 Gradient Boosting Regression\n'
    '(LOO Cross-Validation;  shaded bands = \u00b11 SD)',
    fontweight='bold')
ax.legend(fontsize=9.0, framealpha=0.92)
ax.grid(alpha=0.22)
ax.set_xlim(tr_sizes.min()-1, tr_sizes.max()+1)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
save_fig(fig, 'fig9_learning_curve')

import shutil
shutil.make_archive('/content/paper_figures_600dpi', 'zip',
                    '/content/figures')
files.download('/content/paper_figures_600dpi.zip')
print('\nAll figures packaged and downloaded.')
for f in sorted(os.listdir('/content/figures')):
    kb = os.path.getsize(f'/content/figures/{f}') / 1024
    print(f'  {f:<50}  {kb:6.0f} KB')


Computing learning curve (LOO-CV)...
  Saved: fig9_learning_curve


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


All figures packaged and downloaded.
  fig2_env_distributions.png                             856 KB
  fig2_env_distributions.tiff                         124362 KB
  fig3_tsv_distribution.png                              912 KB
  fig3_tsv_distribution.tiff                          175133 KB
  fig4_loo_predicted_vs_observed.png                     701 KB
  fig4_loo_predicted_vs_observed.tiff                  58652 KB
  fig5_residual_diagnostics.png                          993 KB
  fig5_residual_diagnostics.tiff                       96857 KB
  fig6_shap_bar.png                                      543 KB
  fig6_shap_bar.tiff                                   67429 KB
  fig7_shap_beeswarm.png                                1529 KB
  fig7_shap_beeswarm.tiff                              89177 KB
  fig8_shap_dependence.png                              1201 KB
  fig8_shap_dependence.tiff                           116802 KB
  fig9_learning_curve.png                                432 KB
  

In [17]:
# CELL 8 — Fig. 1: Research Framework Diagram
# Run LAST after Cells 0-7.
# All values read from computed variables.
import subprocess, os, glob, warnings
warnings.filterwarnings('ignore')
subprocess.run(['apt-get','install','-y','msttcorefonts','-qq'],
               capture_output=True)
import matplotlib.font_manager as fm
for f in glob.glob(
        os.path.join(matplotlib.get_cachedir(),'*.json')):
    os.remove(f)
try:
    fm._load_fontmanager(try_read_cache=False)
except Exception:
    pass
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch, Polygon
import numpy as np, os
os.makedirs('/content/figures', exist_ok=True)
names = [f.name for f in fm.fontManager.ttflist]
FONT  = ('Times New Roman' if 'Times New Roman' in names
         else 'DejaVu Serif')

# --- Values from computed variables ---
S_SOLAR = mean_abs_shap_norm[FEATS.index('Rad_mean')]
S_WWR   = mean_abs_shap_norm[FEATS.index('WWR')]
S_PM25  = mean_abs_shap_norm[FEATS.index('PM25_mean')]
S_RH    = mean_abs_shap_norm[FEATS.index('RH_mean')]
S_CO2   = mean_abs_shap_norm[FEATS.index('CO2_mean')]
S_TG    = mean_abs_shap_norm[FEATS.index('TG_mean')]
DOI = 'https://doi.org/10.5281/zenodo.19657430'

C = {'z1f':'#EBF5FB','z1e':'#185FA5','z1d':'#0C447C',
     'z2f':'#EEEDFE','z2e':'#534AB7','z2d':'#3C3489',
     'z3f':'#E1F5EE','z3e':'#1D9E75','z3d':'#085041',
     'z4f':'#EEEDFE','z4e':'#534AB7','z4d':'#3C3489',
     'z5f':'#FEF9E7','z5e':'#BA7517','z5d':'#412402',
     'z6f':'#FAECE7','z6e':'#993C1D','z6d':'#4A1B0C',
     'z7f':'#B03A2E','c1':'#185FA5','c2':'#534AB7',
     'c3':'#7F77DD','c4':'#B4B2A9',
     's1':'#185FA5','s2':'#534AB7',
     's3':'#7F77DD','s4':'#888780',
     'wh':'#FFFFFF','dk':'#1C2833','lt':'#CCCCCC',
     'mu':'#5D6D7E','fa':'#D5D8DC'}

DPI=600; FW,FH=10.0,6.2
fig = plt.figure(figsize=(FW,FH), dpi=DPI, facecolor='white')
ax  = fig.add_axes([0,0,1,1])
ax.set_xlim(0,FW); ax.set_ylim(0,FH); ax.axis('off')
CX=3.50
ZG={1:(5.26,6.00,2.50,2.90),2:(4.58,5.26,2.00,2.50),
    3:(3.76,4.58,1.70,2.00),4:(3.08,3.76,1.40,1.70),
    5:(1.58,3.08,2.10,1.40),6:(0.76,1.58,2.90,2.10)}
OUT_YB=0.20; OUT_H=0.56; RP_X=7.30; RP_W=2.55; RP_YT=6.00

def hw_at(y,zn):
    yb,yt,hwb,hwt=ZG[zn]
    return hwb+(hwt-hwb)*(y-yb)/(yt-yb)
def rbox(ax,x,y,w,h,fc,ec='none',lw=0.5,r=0.06,z=3):
    ax.add_patch(FancyBboxPatch(
        (x,y),w,h,
        boxstyle=f'round,pad=0.01,rounding_size={r}',
        fc=fc,ec=ec,lw=lw,zorder=z))
def pill(ax,xc,yc,w,h,fc,ec='none',lw=0,z=5):
    r=min(h/2*0.90,0.10)
    ax.add_patch(FancyBboxPatch(
        (xc-w/2,yc-h/2),w,h,
        boxstyle=f'round,pad=0.005,rounding_size={r}',
        fc=fc,ec=ec,lw=lw,zorder=z))
def T(ax,x,y,s,col,fs,ha='center',va='center',
      bold=False,italic=False,z=8):
    ax.text(x,y,s,color=col,fontsize=fs,ha=ha,va=va,zorder=z,
            fontweight='bold' if bold else 'normal',
            style='italic' if italic else 'normal',
            fontfamily=FONT)
def hl(ax,y,x0,x1,col=None,lw=0.4):
    ax.plot([x0,x1],[y,y],color=col or C['lt'],lw=lw,zorder=1)

for zn,(yb,yt,hwb,hwt) in ZG.items():
    ax.add_patch(Polygon(
        [(CX-hwt,yt),(CX+hwt,yt),(CX+hwb,yb),(CX-hwb,yb)],
        closed=True, facecolor=C[f'z{zn}f'],
        edgecolor=C[f'z{zn}e'], lw=0.7, zorder=2))

for i,(zn,(yb,yt,_,_)) in enumerate(ZG.items(),start=1):
    T(ax,0.45,(yb+yt)/2,f'{i:02d}',
      [C['c1'],C['c2'],C['z3e'],C['z4e'],
       C['z5e'],C['z6e']][i-1],12,bold=True)

hl(ax,6.17,0.20,9.85,col=C['dk'],lw=1.4)
T(ax,4.90,6.09,
  'Urban Microclimate-Responsive Indoor Thermal Comfort '
  'in Subtropical Elderly Care Facilities',
  C['dk'],9.5,bold=True)
T(ax,4.90,5.96,
  'Guiyang Summer 2025 | 13 Facilities | '
  'Gradient Boosting + SHAP + Evidence-Based Design',
  C['mu'],7.0)
hl(ax,5.88,0.20,9.85,col=C['lt'],lw=0.3)

# Z1 — SHAP chips
yb1,yt1=ZG[1][:2]; CHY=yt1-0.30; hw1=hw_at(CHY,1)
chips=[
    (CX-hw1+0.10,0.88,0.32,C['c1'],
     'Solar radiation',f'{S_SOLAR:.3f}'),
    (CX-hw1+1.06,0.72,0.26,C['c2'],
     'WWR',f'{S_WWR:.3f}'),
    (CX-hw1+1.84,0.64,0.22,C['c2'],
     'PM2.5',f'{S_PM25:.3f}'),
    (CX-hw1+2.54,0.56,0.18,C['c3'],
     'RH',f'{S_RH:.3f}'),
    (CX-hw1+3.16,0.52,0.16,C['c3'],
     'CO2',f'{S_CO2:.3f}'),
    (CX-hw1+3.74,0.50,0.15,C['c4'],
     'TG',f'{S_TG:.3f}'),
]
for xl,w,h,fc,l1,l2 in chips:
    rbox(ax,xl,CHY-h/2,w,h,fc=fc,r=0.04,z=6)
    T(ax,xl+w/2,CHY+h*0.13,l1,C['wh'],7.0,bold=True,z=7)
    T(ax,xl+w/2,CHY-h*0.26,l2,'#C8DCF0',6.0,z=7)
T(ax,CX,yb1+0.22,
  'Urban Microclimate \u00b7 Guiyang \u00b7 5 Districts',
  C['z1d'],8.0,bold=True)
pill(ax,CX,yb1+0.10,1.80,0.14,C['fa'],z=5)
T(ax,CX,yb1+0.10,
  'Ta \u00b7 Wind \u00b7 Noise (SHAP<0.05)',C['mu'],6.5)

# Z2
yb2,yt2=ZG[2][:2]; ym2=(yb2+yt2)/2
T(ax,CX,ym2+0.12,
  'Building Envelope \u00b7 Window-to-Wall Ratio',
  C['z2d'],8.0,bold=True)
T(ax,CX,ym2-0.14,
  'WWR 0.10\u20130.73 (mean 0.36) \u00b7 '
  'CAD-measured \u00b7 13 facilities',
  C['z2e'],7.0)

# Z3
yb3,yt3=ZG[3][:2]; ym3=(yb3+yt3)/2
T(ax,CX,ym3+0.16,
  '168-h Monitoring \u00b7 n=234 Questionnaires (ASHRAE 55-2023)',
  C['z3d'],8.0,bold=True)
T(ax,CX,ym3-0.03,
  'Days 1,4,7 \u00b7 10:00 & 15:00 \u00b7 '
  '3 respondents per time-point',
  C['z3e'],7.0)
hw3p=hw_at(yb3+0.20,3)
for xc,lbl in [(CX-hw3p+0.74,'2,184 records'),
               (CX+0.22,'n=78 merged')]:
    pill(ax,xc,yb3+0.20,1.18,0.24,C['z3e'],z=5)
    T(ax,xc,yb3+0.20,lbl,C['wh'],7.0)

# Z4
yb4,yt4=ZG[4][:2]; ym4=(yb4+yt4)/2
T(ax,CX,ym4+0.14,
  'Gradient Boosting + LOO Cross-Validation',
  C['z4d'],8.0,bold=True)
T(ax,CX,ym4-0.12,
  f'R\u00b2={LOO_R2:.4f} \u00b7 RMSE={LOO_RMSE:.3f} \u00b7 '
  f'MAE={LOO_MAE:.3f} \u00b7 SW p={sw_p:.3f}',
  C['z4e'],7.0)

# Z5
yb5,yt5=ZG[5][:2]
T(ax,CX,yt5-0.20,
  'SHAP Interpretability (TreeExplainer)',
  C['z5d'],8.0,bold=True)
BXL=CX-1.30; BMAX=1.50; BH=0.15
BTOP=yt5-0.50; BGAP=0.22
shap_rows=[
    (1.000,      C['s1'],'Solar radiation',f'{S_SOLAR:.3f}'),
    (S_WWR/S_SOLAR, C['s2'],'WWR',         f'{S_WWR:.3f}'),
    (S_PM25/S_SOLAR,C['s3'],'PM2.5',       f'{S_PM25:.3f}'),
    (S_RH/S_SOLAR,  C['s4'],'Humidity RH', f'{S_RH:.3f}'),
]
for i,(frac,fc,lbl,val) in enumerate(shap_rows):
    bw=frac*BMAX; bcy=BTOP-i*BGAP
    rbox(ax,BXL,bcy-BH/2,bw,BH,fc=fc,r=0.03,z=7)
    T(ax,BXL+bw+0.10,bcy,f'{lbl}   {val}',
      fc,7.0,ha='left',bold=(frac==1.0))
T(ax,BXL+0.1,BTOP-4*BGAP-0.04,
  '+ CO2 \u00b7 TG \u00b7 Ta \u00b7 Noise \u00b7 Wind',
  C['mu'],6.5,ha='left',italic=True)

# Z6 — Evidence-based design recommendation (no NSGA-II)
yb6,yt6=ZG[6][:2]; hw6t=hw_at(yt6-0.12,6)
pill(ax,CX,yt6-0.12,min(2*hw6t-0.20,3.60),0.20,C['z5e'],z=5)
T(ax,CX,yt6-0.12,
  'Solar radiation primary driver \u00b7 '
  'WWR as key controllable parameter',
  C['wh'],6.5)
T(ax,CX,(yb6+yt6)/2+0.10,
  'Evidence-Based Design Recommendation',
  C['z6d'],8.0,bold=True)
T(ax,CX,(yb6+yt6)/2-0.14,
  'WWR \u2264 0.39 \u00b7 Based on 13-institution field data \u00b7 '
  'Guiyang Summer 2025',
  C['z6e'],6.5)

# Bottom output bar
rbox(ax,0.20,OUT_YB,6.60,OUT_H,fc=C['z7f'],r=0.06,z=3)
for xc,txt in [
    (1.00,'WWR \u2264 0.39'),
    (2.36,f'Solar SHAP {S_SOLAR:.3f}'),
    (3.68,'n=13 Sites'),
    (5.00,'Summer 2025')]:
    pill(ax,xc,OUT_YB+0.28,1.08,0.22,C['wh'],z=6)
    T(ax,xc,OUT_YB+0.28,txt,C['z7f'],7.0,bold=True,z=7)
T(ax,3.50,OUT_YB+0.07,
  'Evidence-based retrofit guidelines \u00b7 Healthy China 2030',
  '#F5B7B1',6.0,z=7)

# Right panel
rbox(ax,RP_X,OUT_YB,RP_W,RP_YT-OUT_YB,
     fc='#F8FAFB',ec=C['lt'],lw=0.5,r=0.08,z=2)
rbox(ax,RP_X,RP_YT-0.34,RP_W,0.34,fc=C['dk'],r=0.08,z=4)
RC=RP_X+RP_W/2; RXL=RP_X+0.13; RXR=RP_X+RP_W-0.10
T(ax,RC,RP_YT-0.17,'Data & Results Summary',
  C['wh'],8.0,bold=True)

def psec(y0,title,hc,rows):
    pill(ax,RC,y0-0.12,RP_W-0.22,0.22,hc,z=5)
    T(ax,RC,y0-0.12,title,C['wh'],7.5,bold=True)
    y=y0-0.34
    for lbl,val,vc in rows:
        T(ax,RXL,y,lbl,C['mu'],7.0,ha='left')
        T(ax,RXR,y,val,vc,7.0,ha='right',bold=True)
        y-=0.22
    hl(ax,y-0.04,RP_X+0.10,RP_X+RP_W-0.10,
       col=C['lt'],lw=0.3)
    return y-0.04

py=RP_YT-0.46
py=psec(py,'Study design',C['z3e'],
        [('Facilities','13',C['z3d']),
         ('Monitoring','168 h',C['z3e']),
         ('Questionnaires','n=234',C['z3e']),
         ('Merged means','n=78',C['z3e'])])
py=psec(py-0.14,'Model performance',C['z4e'],
        [(f'LOO R\u00b2',f'{LOO_R2:.4f}',C['z4d']),
         ('RMSE',f'{LOO_RMSE:.3f}',C['z4e']),
         ('MAE',f'{LOO_MAE:.3f}',C['z4e']),
         ('S-W p',f'{sw_p:.3f}',C['z4e'])])
py=psec(py-0.14,'SHAP features',C['z5e'],
        [('Solar rad.',f'{S_SOLAR:.3f}',C['s1']),
         ('WWR',f'{S_WWR:.3f}',C['s2']),
         ('PM2.5',f'{S_PM25:.3f}',C['s3']),
         ('RH',f'{S_RH:.3f}',C['s4'])])
psec(py-0.14,'Design guidance',C['z6e'],
     [('Rec. WWR','\u22640.39',C['z6d']),
      ('Solar SHAP',f'{S_SOLAR:.3f}',C['z6e']),
      ('WWR SHAP',f'{S_WWR:.3f}',C['z6e']),
      ('Data basis','n=78',C['z6e'])])

hl(ax,0.14,0.20,9.85,col=C['lt'],lw=0.3)
T(ax,0.25,0.07,
  '13 facilities \u00b7 Guiyang, Guizhou \u00b7 July-August 2025',
  C['mu'],6.0,ha='left')
T(ax,9.70,0.07, DOI, C['mu'],6.0,ha='right')

fig.savefig('/content/figures/fig1_framework.png',
            dpi=600,bbox_inches='tight',facecolor='white')
fig.savefig('/content/figures/fig1_framework.tiff',
            dpi=600,bbox_inches='tight',facecolor='white')
plt.close()
print('Fig. 1 saved.')
files.download('/content/figures/fig1_framework.png')

Fig. 1 saved.


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>